# Functional Autoencoder Training for Diffusion Models

This notebook implements the **Encoder** and **Decoder** for functional data generation using latent space diffusion models.

## Architecture Overview

1. **Encoder**: Maps functional data `x(t)` (with any discretization) → latent vector `z ∈ R^d`
2. **Decoder**: Maps latent vector `z` + query points `t_q` → reconstructed function `x'(t_q)`

This approach is based on the FunDiff paper and enables:
- Independence from discretization (can train on 100 points, generate on 200)
- Efficient diffusion in low-dimensional latent space
- Minimax-optimal density estimation for functional data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import os

# Import scikit-fda
from skfda.representation.grid import FDataGrid
from skfda.datasets import make_sinusoidal_process

## 1. Generate Toy Functional Dataset

We create synthetic functional data with varying amplitude, frequency, and phase:
$$x(t) = A \cdot \sin(\omega t + \phi) + \epsilon(t)$$

In [ ]:
def generate_toy_functional_data(n_samples=1000, n_points=100, domain_range=(0, 1)):
    """
    Generate synthetic functional data with varying parameters.
    
    Args:
        n_samples: Number of functional samples to generate
        n_points: Number of discretization points
        domain_range: Tuple (start, end) for the domain
    
    Returns:
        FDataGrid object containing the synthetic functions
    """
    t = np.linspace(domain_range[0], domain_range[1], n_points)
    data_matrix = np.zeros((n_samples, n_points, 1))
    
    for i in range(n_samples):
        # Random amplitude, frequency, and phase
        A = np.random.uniform(0.5, 2.0)
        omega = np.random.uniform(2 * np.pi, 6 * np.pi)
        phi = np.random.uniform(0, 2 * np.pi)
        
        # Generate function with small noise
        noise = np.random.normal(0, 0.05, n_points)
        x_t = A * np.sin(omega * t + phi) + noise
        data_matrix[i, :, 0] = x_t
    
    return FDataGrid(data_matrix=data_matrix, grid_points=t)

# Generate training data
print("Generating toy functional data...")
n_train = 800
n_val = 200
n_points = 100

fdata_train = generate_toy_functional_data(n_samples=n_train, n_points=n_points)
fdata_val = generate_toy_functional_data(n_samples=n_val, n_points=n_points)

print(f"Training data shape: {fdata_train.data_matrix.shape}")
print(f"Validation data shape: {fdata_val.data_matrix.shape}")

# Visualize some samples
fig, ax = plt.subplots(figsize=(10, 4))
for i in range(10):
    ax.plot(fdata_train.grid_points[0], fdata_train.data_matrix[i, :, 0], alpha=0.7)
ax.set_xlabel('t')
ax.set_ylabel('x(t)')
ax.set_title('Sample Synthetic Functional Data')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Define the Functional Encoder

The encoder uses 1D-CNN followed by Global Average Pooling to create a fixed-size latent representation, independent of the number of input points.

In [ ]:
class FunctionalEncoder(nn.Module):
    """Encoder network for functional data.
    
    Maps functional data x(t) with any discretization to a fixed-size latent vector z.
    Uses 1D convolutions followed by global average pooling for discretization independence.
    """
    
    def __init__(self, latent_dim=64, n_features=1):
        super().__init__()
        self.latent_dim = latent_dim
        
        # 1D Convolutional layers
        self.conv1 = nn.Conv1d(n_features, 32, kernel_size=7, padding=3)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=5, padding=2)
        self.conv3 = nn.Conv1d(64, 128, kernel_size=3, padding=1)
        self.conv4 = nn.Conv1d(128, 128, kernel_size=3, padding=1)
        
        # Batch normalization
        self.bn1 = nn.BatchNorm1d(32)
        self.bn2 = nn.BatchNorm1d(64)
        self.bn3 = nn.BatchNorm1d(128)
        self.bn4 = nn.BatchNorm1d(128)
        
        # Global pooling - makes output independent of input length
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        
        # Final projection to latent space
        self.fc = nn.Linear(128, latent_dim)
    
    def forward(self, x):
        """
        Args:
            x: Tensor of shape (batch, n_points, n_features)
        
        Returns:
            z: Latent vector of shape (batch, latent_dim)
        """
        # Transpose to (batch, n_features, n_points) for Conv1d
        x = x.transpose(1, 2)
        
        # Convolutional layers with ReLU and batch norm
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        x = F.relu(self.bn4(self.conv4(x)))
        
        # Global pooling: (batch, 128, n_points) -> (batch, 128, 1)
        x = self.global_pool(x)
        
        # Flatten and project to latent space
        x = x.squeeze(-1)  # (batch, 128)
        z = self.fc(x)     # (batch, latent_dim)
        
        return z

# Test the encoder
encoder = FunctionalEncoder(latent_dim=64)
test_input = torch.randn(4, 100, 1)  # batch_size=4, n_points=100, n_features=1
test_output = encoder(test_input)
print(f"Encoder test: Input shape {test_input.shape} -> Output shape {test_output.shape}")

## 3. Define the Functional Decoder (Neural Implicit Representation)

The decoder is a **Neural Implicit Representation (INR)** that takes:
- Latent vector `z`
- Query points `t_q`

And outputs the function value at those points. This allows generation at arbitrary discretizations.

In [ ]:
class FunctionalDecoder(nn.Module):
    """Decoder network using Neural Implicit Representation (INR).
    
    Maps latent vector z and query points t_q to function values x(t_q).
    This allows evaluation at arbitrary discretizations.
    """
    
    def __init__(self, latent_dim=64, hidden_dim=128, n_features=1):
        super().__init__()
        self.latent_dim = latent_dim
        
        # MLP that processes concatenation of z and t
        # Input: latent_dim + 1 (for time coordinate)
        self.fc1 = nn.Linear(latent_dim + 1, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, hidden_dim)
        self.fc4 = nn.Linear(hidden_dim, n_features)
        
        # Batch normalization
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.bn2 = nn.BatchNorm1d(hidden_dim)
        self.bn3 = nn.BatchNorm1d(hidden_dim)
    
    def forward(self, z, t_query):
        """
        Args:
            z: Latent vector of shape (batch, latent_dim)
            t_query: Query points of shape (batch, n_points_query, 1) or (batch, n_points_query)
        
        Returns:
            x_recon: Reconstructed function values of shape (batch, n_points_query, n_features)
        """
        batch_size = z.shape[0]
        
        # Ensure t_query has shape (batch, n_points_query, 1)
        if t_query.dim() == 2:
            t_query = t_query.unsqueeze(-1)
        
        n_points_query = t_query.shape[1]
        
        # Expand z to match query points: (batch, latent_dim) -> (batch, n_points_query, latent_dim)
        z_expanded = z.unsqueeze(1).expand(-1, n_points_query, -1)
        
        # Concatenate z and t_query: (batch, n_points_query, latent_dim + 1)
        inp = torch.cat([z_expanded, t_query], dim=-1)
        
        # Reshape for batch norm: (batch * n_points_query, latent_dim + 1)
        inp = inp.reshape(-1, self.latent_dim + 1)
        
        # MLP processing
        x = F.relu(self.bn1(self.fc1(inp)))
        x = F.relu(self.bn2(self.fc2(x)))
        x = F.relu(self.bn3(self.fc3(x)))
        x = self.fc4(x)
        
        # Reshape back: (batch * n_points_query, n_features) -> (batch, n_points_query, n_features)
        x = x.reshape(batch_size, n_points_query, -1)
        
        return x

# Test the decoder
decoder = FunctionalDecoder(latent_dim=64)
test_z = torch.randn(4, 64)  # batch_size=4, latent_dim=64
test_t = torch.linspace(0, 1, 100).unsqueeze(0).expand(4, -1)  # batch_size=4, n_points=100
test_recon = decoder(test_z, test_t)
print(f"Decoder test: Latent shape {test_z.shape}, Query points shape {test_t.shape} -> Output shape {test_recon.shape}")

## 4. Create PyTorch Dataset and DataLoader

In [ ]:
class FunctionalDataset(Dataset):
    """PyTorch Dataset for functional data."""
    
    def __init__(self, fdata):
        """
        Args:
            fdata: FDataGrid object
        """
        self.data_matrix = torch.FloatTensor(fdata.data_matrix)
        self.grid_points = torch.FloatTensor(fdata.grid_points[0])
    
    def __len__(self):
        return len(self.data_matrix)
    
    def __getitem__(self, idx):
        return self.data_matrix[idx], self.grid_points

# Create datasets and dataloaders
train_dataset = FunctionalDataset(fdata_train)
val_dataset = FunctionalDataset(fdata_val)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

## 5. Train the Autoencoder

In [ ]:
def train_autoencoder(encoder, decoder, train_loader, val_loader, n_epochs=100, lr=1e-3, device='cpu'):
    """
    Train the functional autoencoder.
    
    Args:
        encoder: FunctionalEncoder model
        decoder: FunctionalDecoder model
        train_loader: Training data loader
        val_loader: Validation data loader
        n_epochs: Number of training epochs
        lr: Learning rate
        device: Device to train on ('cpu' or 'cuda')
    
    Returns:
        Dictionary with training history
    """
    encoder = encoder.to(device)
    decoder = decoder.to(device)
    
    # Optimizer for both encoder and decoder
    optimizer = torch.optim.Adam(
        list(encoder.parameters()) + list(decoder.parameters()),
        lr=lr
    )
    
    # Learning rate scheduler
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=10, verbose=True
    )
    
    history = {'train_loss': [], 'val_loss': []}
    best_val_loss = float('inf')
    
    for epoch in range(n_epochs):
        # Training phase
        encoder.train()
        decoder.train()
        train_loss = 0.0
        
        for data, grid_points in train_loader:
            data = data.to(device)
            grid_points = grid_points.to(device)
            
            optimizer.zero_grad()
            
            # Forward pass
            z = encoder(data)
            recon = decoder(z, grid_points)
            
            # Reconstruction loss (MSE)
            loss = F.mse_loss(recon, data)
            
            # Backward pass
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * len(data)
        
        train_loss /= len(train_loader.dataset)
        
        # Validation phase
        encoder.eval()
        decoder.eval()
        val_loss = 0.0
        
        with torch.no_grad():
            for data, grid_points in val_loader:
                data = data.to(device)
                grid_points = grid_points.to(device)
                
                z = encoder(data)
                recon = decoder(z, grid_points)
                loss = F.mse_loss(recon, data)
                
                val_loss += loss.item() * len(data)
        
        val_loss /= len(val_loader.dataset)
        
        # Update learning rate
        scheduler.step(val_loss)
        
        # Save history
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        
        # Print progress
        if (epoch + 1) % 10 == 0:
            print(f"Epoch [{epoch+1}/{n_epochs}] - Train Loss: {train_loss:.6f}, Val Loss: {val_loss:.6f}")
        
        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save({
                'encoder_state_dict': encoder.state_dict(),
                'decoder_state_dict': decoder.state_dict(),
                'epoch': epoch,
                'val_loss': val_loss
            }, 'best_autoencoder.pth')
    
    return history

# Initialize models
latent_dim = 64
encoder = FunctionalEncoder(latent_dim=latent_dim)
decoder = FunctionalDecoder(latent_dim=latent_dim)

# Check for GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on device: {device}")

# Train the autoencoder
print("\nStarting training...")
history = train_autoencoder(
    encoder, decoder,
    train_loader, val_loader,
    n_epochs=100,
    lr=1e-3,
    device=device
)

## 6. Visualize Training Progress

In [ ]:
# Plot training history
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(history['train_loss'], label='Training Loss', linewidth=2)
ax.plot(history['val_loss'], label='Validation Loss', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('Autoencoder Training Progress')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Evaluate Reconstruction Quality

In [ ]:
# Load best model
checkpoint = torch.load('best_autoencoder.pth', map_location=device)
encoder.load_state_dict(checkpoint['encoder_state_dict'])
decoder.load_state_dict(checkpoint['decoder_state_dict'])
encoder.eval()
decoder.eval()

# Test reconstruction on validation data
with torch.no_grad():
    # Get a batch of validation data
    data_batch, grid_points = next(iter(val_loader))
    data_batch = data_batch.to(device)
    grid_points = grid_points.to(device)
    
    # Encode and decode
    z = encoder(data_batch)
    recon = decoder(z, grid_points)
    
    # Move to CPU for plotting
    data_batch = data_batch.cpu()
    recon = recon.cpu()
    grid_points = grid_points.cpu()

# Visualize reconstruction
n_samples_to_plot = 5
fig, axes = plt.subplots(n_samples_to_plot, 1, figsize=(12, 2*n_samples_to_plot))

for i in range(n_samples_to_plot):
    ax = axes[i]
    ax.plot(grid_points.numpy(), data_batch[i, :, 0].numpy(), 
            label='Original', linewidth=2, alpha=0.7)
    ax.plot(grid_points.numpy(), recon[i, :, 0].numpy(), 
            label='Reconstructed', linewidth=2, linestyle='--', alpha=0.7)
    ax.set_xlabel('t')
    ax.set_ylabel('x(t)')
    ax.set_title(f'Sample {i+1}: Original vs Reconstructed')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Test Discretization Independence

A key feature of our INR decoder is that it can evaluate at arbitrary discretizations.

In [ ]:
# Test with different discretizations
with torch.no_grad():
    # Get one sample
    sample_idx = 0
    data_sample = data_batch[sample_idx:sample_idx+1].to(device)
    
    # Encode once
    z_sample = encoder(data_sample)
    
    # Decode at different resolutions
    t_50 = torch.linspace(0, 1, 50).unsqueeze(0).to(device)
    t_100 = torch.linspace(0, 1, 100).unsqueeze(0).to(device)
    t_200 = torch.linspace(0, 1, 200).unsqueeze(0).to(device)
    
    recon_50 = decoder(z_sample, t_50).cpu()
    recon_100 = decoder(z_sample, t_100).cpu()
    recon_200 = decoder(z_sample, t_200).cpu()

# Plot
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(grid_points.numpy(), data_batch[sample_idx, :, 0].numpy(), 
        'o-', label='Original (100 points)', linewidth=2, markersize=3)
ax.plot(t_50.cpu().numpy()[0], recon_50[0, :, 0].numpy(), 
        's-', label='Reconstructed (50 points)', linewidth=2, markersize=4, alpha=0.7)
ax.plot(t_200.cpu().numpy()[0], recon_200[0, :, 0].numpy(), 
        '^-', label='Reconstructed (200 points)', linewidth=1, markersize=2, alpha=0.7)
ax.set_xlabel('t')
ax.set_ylabel('x(t)')
ax.set_title('Discretization Independence: Same Latent Code, Different Resolutions')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nThe decoder can evaluate the same latent representation at any discretization!")

## 9. Save Trained Models

We save the encoder separately for use in the diffusion model training.

In [ ]:
# Create models directory if it doesn't exist
os.makedirs('models', exist_ok=True)

# Save encoder for latent space diffusion
torch.save({
    'encoder_state_dict': encoder.state_dict(),
    'latent_dim': latent_dim,
    'n_features': 1
}, 'models/functional_encoder.pth')

# Save complete autoencoder
torch.save({
    'encoder_state_dict': encoder.state_dict(),
    'decoder_state_dict': decoder.state_dict(),
    'latent_dim': latent_dim,
    'n_features': 1,
    'val_loss': checkpoint['val_loss']
}, 'models/functional_autoencoder.pth')

print("Models saved successfully!")
print("  - models/functional_encoder.pth (for diffusion training)")
print("  - models/functional_autoencoder.pth (complete autoencoder)")

## Summary

In this notebook, we:

1. ✅ Generated synthetic functional data with varying parameters
2. ✅ Implemented a **Functional Encoder** using 1D-CNNs with global pooling
3. ✅ Implemented a **Functional Decoder** using Neural Implicit Representations (INR)
4. ✅ Trained the autoencoder to reconstruct functional data
5. ✅ Demonstrated discretization independence
6. ✅ Saved the encoder for the next phase: **Latent Space Diffusion**

### Next Steps

Proceed to `02_latent_diffusion_training.ipynb` to:
- Encode all training data into latent space
- Train a diffusion model (SDE/DDPM) in the low-dimensional latent space
- Generate new latent codes from noise